# DX 704 Week 9 Project

This week's project will build an email spam classifier based on the Enron email data set.
You will perform your own feature extraction, and use naive Bayes to estimate the probability that a particular email is spam or not.
Finally, you will review the tradeoffs from different thresholds for automatically sending emails to the junk folder.

The full project description and a template notebook are available on GitHub: [Project 9 Materials](https://github.com/bu-cds-dx704/dx704-project-09).


## Example Code

You may find it helpful to refer to these GitHub repositories of Jupyter notebooks for example code.

* https://github.com/bu-cds-omds/dx601-examples
* https://github.com/bu-cds-omds/dx602-examples
* https://github.com/bu-cds-omds/dx603-examples
* https://github.com/bu-cds-omds/dx704-examples

Any calculations demonstrated in code examples or videos may be found in these notebooks, and you are allowed to copy this example code in your homework answers.

## Part 1: Download Data Set

We will be using the Enron spam data set as prepared in this GitHub repository.

https://github.com/MWiechmann/enron_spam_data

You may need to download this differently depending on your environment.

In [4]:
!wget https://github.com/MWiechmann/enron_spam_data/raw/refs/heads/master/enron_spam_data.zip

'wget' is not recognized as an internal or external command,
operable program or batch file.


In [5]:
import pandas as pd

In [6]:
# pandas can read the zip file directly
enron_spam_data = pd.read_csv("enron_spam_data.zip")
enron_spam_data

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,NaN,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
...,...,...,...,...,...
33711,33711,= ? iso - 8859 - 1 ? q ? good _ news _ c = eda...,"hello , welcome to gigapharm onlinne shop .\np...",spam,2005-07-29
33712,33712,all prescript medicines are on special . to be...,i got it earlier than expected and it was wrap...,spam,2005-07-29
33713,33713,the next generation online pharmacy .,are you ready to rock on ? let the man in you ...,spam,2005-07-30
33714,33714,bloow in 5 - 10 times the time,learn how to last 5 - 10 times longer in\nbed ...,spam,2005-07-30


In [7]:
(enron_spam_data["Spam/Ham"] == "spam").mean()

np.float64(0.5092834262664611)

## Part 2: Design a Feature Extractor

Design a feature extractor for this data set and write out two files of features based on the text.
Don't forget that both the Subject and Message columns are relevant sources of text data.
For each email, you should count the number of repetitions of each feature present.
The auto-grader will assume that you are using a multinomial distribution in the following problems.

In [ ]:
# I will extract bigrams for the feature extractor using bag of words
from sklearn.feature_extraction.text import CountVectorizer
vectorizer = CountVectorizer(ngram_range=(2, 2))

In [12]:
sum(enron_spam_data['Subject'].isna())

289

In [13]:
sum(enron_spam_data['Message'].isna())

371

In [17]:
df_cleaned = enron_spam_data.copy()

In [18]:
df_cleaned["Subject"] = df_cleaned["Subject"].fillna("")
df_cleaned["Message"] = df_cleaned["Message"].fillna("")
df_cleaned

,Message ID,Subject,Message,Spam/Ham,Date
0,0,christmas tree farm pictures,,ham,1999-12-10
1,1,"vastar resources , inc .","gary , production from the high island larger ...",ham,1999-12-13
2,2,calpine daily gas nomination,- calpine daily gas nomination 1 . doc,ham,1999-12-14
3,3,re : issue,fyi - see note below - already done .\nstella\...,ham,1999-12-14
4,4,meter 7268 nov allocation,fyi .\n- - - - - - - - - - - - - - - - - - - -...,ham,1999-12-14
...,...,...,...,...,...
33711,33711,= ? iso - 8859 - 1 ? q ? good _ news _ c = eda...,"hello , welcome to gigapharm onlinne shop .\np...",spam,2005-07-29
33712,33712,all prescript medicines are on special . to be...,i got it earlier than expected and it was wrap...,spam,2005-07-29
33713,33713,the next generation online pharmacy .,are you ready to rock on ? let the man in you ...,spam,2005-07-30
33714,33714,bloow in 5 - 10 times the time,learn how to last 5 - 10 times longer in\nbed ...,spam,2005-07-30


In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer(
    transformers=[
        ('Subject', vectorizer, 'Subject'),
        ('Message', vectorizer, 'Message')
    ]
)

X_transformed = preprocessor.fit_transform(df_cleaned)
X_transformed

<Compressed Sparse Row sparse matrix of dtype 'int64'
	with 4074079 stored elements and shape (33716, 173054)>

In [31]:
feature_names = preprocessor.get_feature_names_out()

In [38]:
import json
features_json = []

for row_number in range(X_transformed.shape[0]):
    row = X_transformed.getrow(row_number)

    # Transform sparse vector into dict
    row_features = {}
    for column_number, count in zip(row.indices, row.data):
        feature_name = feature_names[column_number]
        row_features[feature_name] = int(count)

    # Convert dict to json dump
    features_json.append(json.dumps(row_features))

In [39]:
features_df = pd.DataFrame({
    "Message ID": enron_spam_data["Message ID"],
    "features_json": features_json,
})
features_df

,Message ID,features_json
0,0,"{""Subject__christmas"": 1, ""Subject__tree"": 1, ..."
1,1,"{""Subject__vastar"": 1, ""Subject__resources"": 1..."
2,2,"{""Subject__calpine"": 1, ""Subject__daily"": 1, ""..."
3,3,"{""Subject__re"": 1, ""Subject__issue"": 1, ""Messa..."
4,4,"{""Subject__meter"": 1, ""Subject__7268"": 1, ""Sub..."
...,...,...
33711,33711,"{""Subject__news"": 1, ""Subject__good"": 1, ""Subj..."
33712,33712,"{""Subject__back"": 1, ""Subject__on"": 1, ""Subjec..."
33713,33713,"{""Subject__the"": 1, ""Subject__next"": 1, ""Subje..."
33714,33714,"{""Subject__the"": 1, ""Subject__in"": 1, ""Subject..."


Assign a row to the test data set if `Message ID % 30 == 0` and assign it to the training data set otherwise.
Write two files, "train-features.tsv" and "test-features.tsv" with two columns, Message ID and features_json.
The features_json column should contain a JSON dictionary where the keys are your feature names and the values are integer feature values.
This will give us a sparse feature representation.


In [40]:
test_mask = features_df["Message ID"] % 30 == 0

In [44]:
features_train_df = features_df.loc[~test_mask].copy()
features_train_df

,Message ID,features_json
1,1,"{""Subject__vastar"": 1, ""Subject__resources"": 1..."
2,2,"{""Subject__calpine"": 1, ""Subject__daily"": 1, ""..."
3,3,"{""Subject__re"": 1, ""Subject__issue"": 1, ""Messa..."
4,4,"{""Subject__meter"": 1, ""Subject__7268"": 1, ""Sub..."
5,5,"{""Subject__gas"": 1, ""Subject__mcmullen"": 1, ""S..."
...,...,...
33711,33711,"{""Subject__news"": 1, ""Subject__good"": 1, ""Subj..."
33712,33712,"{""Subject__back"": 1, ""Subject__on"": 1, ""Subjec..."
33713,33713,"{""Subject__the"": 1, ""Subject__next"": 1, ""Subje..."
33714,33714,"{""Subject__the"": 1, ""Subject__in"": 1, ""Subject..."


In [45]:
features_test_df = features_df.loc[test_mask].copy()
features_test_df

,Message ID,features_json
0,0,"{""Subject__christmas"": 1, ""Subject__tree"": 1, ..."
30,30,"{""Subject__january"": 1, ""Subject__production"":..."
60,60,"{""Subject__new"": 1, ""Subject__pooling"": 1, ""Su..."
90,90,"{""Subject__for"": 1, ""Subject__january"": 1, ""Su..."
120,120,"{""Subject__re"": 1, ""Subject__meter"": 1, ""Subje..."
...,...,...
33570,33570,"{""Subject__buy"": 1, ""Subject__drugs"": 1, ""Subj..."
33600,33600,"{""Subject__of"": 1, ""Subject__the"": 1, ""Subject..."
33630,33630,"{""Subject__cheap"": 1, ""Subject__medication"": 1..."
33660,33660,"{""Subject__up"": 1, ""Subject__to"": 1, ""Subject_..."


Submit "train-features.tsv" and "test-features.tsv" in Gradescope.

In [46]:
features_train_df.to_csv("train-features.tsv", sep="\t", index=False)
features_test_df.to_csv("test-features.tsv", sep="\t", index=False)

Hint: these features will be graded based on the test accuracy of a logistic regression based on the training features.
This is to make sure that your feature set is not degenerate; you do not need to compute this regression yourself.
You can separately assess your feature quality based on your results in part 6.

## Part 3: Compute Conditional Probabilities

Based on your training data, compute appropriate conditional probabilities for use with naïve Bayes.
Use of additive smoothing with $\alpha=1$ to avoid zeros.


In [48]:
alpha = 1.0

In [50]:
y = enron_spam_data.set_index("Message ID")["Spam/Ham"]
y

Message ID
0         ham
1         ham
2         ham
3         ham
4         ham
         ... 
33711    spam
33712    spam
33713    spam
33714    spam
33715    spam
Name: Spam/Ham, Length: 33716, dtype: str

In [54]:
features_train_df

,Message ID,features_json
1,1,"{""Subject__vastar"": 1, ""Subject__resources"": 1..."
2,2,"{""Subject__calpine"": 1, ""Subject__daily"": 1, ""..."
3,3,"{""Subject__re"": 1, ""Subject__issue"": 1, ""Messa..."
4,4,"{""Subject__meter"": 1, ""Subject__7268"": 1, ""Sub..."
5,5,"{""Subject__gas"": 1, ""Subject__mcmullen"": 1, ""S..."
...,...,...
33711,33711,"{""Subject__news"": 1, ""Subject__good"": 1, ""Subj..."
33712,33712,"{""Subject__back"": 1, ""Subject__on"": 1, ""Subjec..."
33713,33713,"{""Subject__the"": 1, ""Subject__next"": 1, ""Subje..."
33714,33714,"{""Subject__the"": 1, ""Subject__in"": 1, ""Subject..."


In [57]:
y_train = y.loc[
    ~test_mask
].reset_index(drop=True)
y_train

0         ham
1         ham
2         ham
3         ham
4         ham
         ... 
32587    spam
32588    spam
32589    spam
32590    spam
32591    spam
Name: Spam/Ham, Length: 32592, dtype: str

In [ ]:
from collections import Counter

# Separately extract probs for each class/feature combination
class_feature_counts = {"ham": Counter(), "spam": Counter()}

# Iterate over each row in the training dataset
# We grab the counts of each label+feature combination for that row and add them to the counts dict
# This effectively gives us counts over the whole training set of how many times we see a label+feature combination
for label, features_json in zip(y_train, features_train_df["features_json"]):
    class_feature_counts[label].update(json.loads(features_json))
class_feature_counts

{'ham': Counter({'Message__the': 176113,
          'Message__to': 125071,
          'Message__and': 80179,
          'Message__of': 74565,
          'Message__in': 58551,
          'Message__enron': 58011,
          'Message__for': 46202,
          'Message__on': 39887,
          'Message__is': 37008,
          'Message__that': 34463,
          'Message__ect': 34399,
          'Message__you': 33231,
          'Message__this': 28034,
          'Message__be': 26748,
          'Message__with': 26219,
          'Message__will': 24659,
          'Message__we': 24492,
          'Message__have': 23556,
          'Message__it': 22512,
          'Message__from': 22005,
          'Message__as': 21729,
          'Message__at': 21337,
          'Message__by': 19690,
          'Message__are': 19279,
          'Message__hou': 16523,
          'Message__or': 15406,
          'Message__if': 14963,
          'Message__2001': 14459,
          'Message__not': 13954,
          'Message__your': 12908,
    

In [60]:
vocabulary = sorted(set(class_feature_counts["ham"]) | set(class_feature_counts["spam"]))
vocab_size = len(vocabulary)
class_feature_totals = {
    label: sum(feature_counts.values())
    for label, feature_counts in class_feature_counts.items()
}

In [62]:
feature_probs = pd.DataFrame({
    "feature": vocabulary,
    "ham_probability": [
        (class_feature_counts["ham"][feature] + alpha)
        / (class_feature_totals["ham"] + alpha * vocab_size)
        for feature in vocabulary
    ],
    "spam_probability": [
        (class_feature_counts["spam"][feature] + alpha)
        / (class_feature_totals["spam"] + alpha * vocab_size)
        for feature in vocabulary
    ],
})

feature_probs

,feature,ham_probability,spam_probability
0,Message__00,1.461227e-03,1.697723e-03
1,Message__000,1.057780e-03,1.535449e-03
2,Message__0000,6.924153e-06,1.441132e-04
3,Message__000000,1.384831e-06,4.452276e-05
4,Message__00000000,1.384831e-06,2.929129e-07
...,...,...,...
170432,Subject__þpensó,2.308051e-07,5.858258e-07
170433,Subject__þquieres,2.308051e-07,5.858258e-07
170434,Subject__þá,2.308051e-07,5.858258e-07
170435,Subject__þãõ,2.308051e-07,1.171652e-06


Save the conditional probabilities in a file "feature-probabilities.tsv" with columns feature, ham_probability and spam_probability.

In [ ]:
feature_probs.to_csv("feature-probabilities.tsv", sep="\t", index=False)

Submit "feature-probabilities.tsv" in Gradescope.

## Part 4: Implement a Naïve Bayes Classifier

Implement a naïve Bayes classifier based on your previous feature probabilities.

In [ ]:
from sklearn.naive_bayes import MultinomialNB

model = MultinomialNB(alpha=1.0)

In [ ]:
model.fit(features_train_df, )

Save your prediction probabilities to "train-predictions.tsv" with columns Message ID, ham and spam.

In [ ]:
# YOUR CHANGES HERE

...

Submit "train-predictions.tsv" in Gradescope.

## Part 5: Predict Spam Probability for Test Data

Use your previous classifier to predict spam probability for the test data.
If the test data has tokens that you did not model, just ignore them.

In [ ]:
# YOUR CHANGES HERE

...

Save your prediction probabilities in "test-predictions.tsv" with the same columns as "train-predictions.tsv".

In [ ]:
# YOUR CHANGES HERE

...

Submit "test-predictions.tsv" in Gradescope.

## Part 6: Construct ROC Curve

For every probability threshold from 0.01 to .99 in increments of 0.01, compute the false and true positive rates from the test data using the spam class for positives.
That is, if the predicted spam probability is greater than or equal to the threshold, predict spam.

In [ ]:
# YOUR CHANGES HERE

...

Save this data in a file "roc.tsv" with columns threshold, false_positive_rate and true_positive rate.

In [ ]:
# YOUR CHANGES HERE

...

Submit "roc.tsv" in Gradescope.

## Part 7: OpenAI API Key

Sign up for an OpenAI API acount at https://platform.openai.com/.
If you already have an OpenAI account (e.g. for ChatGPT), you may reuse that account.

Create a "dx704" project at https://platform.openai.com/settings/organization/projects, setup a $10/month spend limit, and create an API key for the project at https://platform.openai.com/api-keys.
You will use this API key for projects in later weeks.


No submission needed.

## Part 8: Code

Please submit a Jupyter notebook that can reproduce all your calculations and recreate the previously submitted files.
You do not need to provide code for data collection if you did that by manually.

## Part 9: Acknowledgements

If you discussed this assignment with anyone, please acknowledge them here.
If you did this assignment completely on your own, simply write none below.

If you used any libraries not mentioned in this module's content, please list them with a brief explanation what you used them for. If you did not use any other libraries, simply write none below.

If you used any generative AI tools, please add links to your transcripts below, and any other information that you feel is necessary to comply with the generative AI policy. If you did not use any generative AI tools, simply write none below.